In [8]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [9]:
# 1. Carga y preparación (misma semilla para consistencia)
df = pd.read_csv("./datasets/crop_recomendation/Crop_recommendation.xls")
X = df.drop("label", axis=1)
y = df["label"]

Y_onehot = pd.get_dummies(y, dtype=float)
crop_names = Y_onehot.columns.to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y_onehot, test_size=0.20, stratify=y, random_state=42
)

modelos = {
    "Regresión Lineal Múltiple": make_pipeline(
        StandardScaler(), LinearRegression()
    ),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting Regressor": MultiOutputRegressor(
        GradientBoostingRegressor(random_state=42)
    ),
    "KNN Regressor (k=5)": make_pipeline(
        StandardScaler(), KNeighborsRegressor(n_neighbors=5)
    ),
}

# ==============================================================================
# 2. ANÁLISIS DE SOBREAJUSTE (Train vs. Test Accuracy)
# ==============================================================================
print("=== 1. ANÁLISIS DE SOBREAJUSTE (TRAIN VS TEST) ===")
overfit_rows = []
modelos_entrenados = {}

for nombre, modelo in modelos.items():
  modelo.fit(X_train, Y_train)
  modelos_entrenados[nombre] = modelo

  # Predicción en Train
  train_pred = crop_names[np.argmax(modelo.predict(X_train), axis=1)]
  train_real = crop_names[np.argmax(Y_train.to_numpy(), axis=1)]
  acc_train = accuracy_score(train_real, train_pred)

  # Predicción en Test
  test_pred = crop_names[np.argmax(modelo.predict(X_test), axis=1)]
  test_real = crop_names[np.argmax(Y_test.to_numpy(), axis=1)]
  acc_test = accuracy_score(test_real, test_pred)

  brecha = acc_train - acc_test
  diagnostico = (
      "Subajuste (Underfitting)"
      if acc_train < 0.80
      else ("Sobreajuste leve" if brecha > 0.03 else "Buen ajuste (Generaliza)")
  )

  overfit_rows.append({
      "Modelo": nombre,
      "Train Accuracy": f"{acc_train:.2%}",
      "Test Accuracy": f"{acc_test:.2%}",
      "Brecha (Train - Test)": f"{brecha:.2%}",
      "Diagnóstico": diagnostico,
  })

print(pd.DataFrame(overfit_rows).to_markdown(index=False))

# ==============================================================================
# 3. ANÁLISIS DE ERRORES (Comparando Regresión Lineal vs. Mejor Modelo)
# ==============================================================================
print("\n=== 2. ANÁLISIS DE ERRORES EN TEST ===")
real_test = crop_names[np.argmax(Y_test.to_numpy(), axis=1)]

for nombre in ["Regresión Lineal Múltiple", "Random Forest Regressor"]:
  pred_test = crop_names[
      np.argmax(modelos_entrenados[nombre].predict(X_test), axis=1)
  ]
  errores = pd.DataFrame({"Real": real_test, "Predicho": pred_test})
  fallos = errores[errores["Real"] != errores["Predicho"]]

  print(
      f"\nModelo: {nombre} | Total errores: {len(fallos)} de {len(real_test)}"
      " muestras"
  )
  if len(fallos) > 0:
    top_confusiones = (
        fallos.value_counts().head(5).reset_index(name="Cantidad de errores")
    )
    print(top_confusiones.to_markdown(index=False))

# ==============================================================================
# 4. IMPORTANCIA DE LAS VARIABLES (Feature Importance del Mejor Modelo)
# ==============================================================================
print("\n=== 3. IMPORTANCIA DE LAS VARIABLES (RANDOM FOREST) ===")
mejor_modelo = modelos_entrenados["Random Forest Regressor"]

importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia (%)": mejor_modelo.feature_importances_ * 100,
}).sort_values(by="Importancia (%)", ascending=False)

print(importancias.to_markdown(index=False, floatfmt=".2f"))

# ==============================================================================
# 5. GUARDAR EL MEJOR MODELO CON JOBLIB
# ==============================================================================
# Se guarda un diccionario con el modelo y los nombres de las columnas objetivo
# para poder decodificar np.argmax() al cargar el archivo en el futuro.
paquete_modelo = {
    "modelo": mejor_modelo,
    "crop_names": crop_names,
    "features": list(X.columns),
}

ruta_archivo = "modelo_supervisado.joblib"
joblib.dump(paquete_modelo, ruta_archivo)
print(f"\nModelo guardado exitosamente en: {ruta_archivo}")


=== 1. ANÁLISIS DE SOBREAJUSTE (TRAIN VS TEST) ===
| Modelo                      | Train Accuracy   | Test Accuracy   | Brecha (Train - Test)   | Diagnóstico              |
|:----------------------------|:-----------------|:----------------|:------------------------|:-------------------------|
| Regresión Lineal Múltiple   | 72.10%           | 72.95%          | -0.85%                  | Subajuste (Underfitting) |
| Random Forest Regressor     | 100.00%          | 99.32%          | 0.68%                   | Buen ajuste (Generaliza) |
| Gradient Boosting Regressor | 100.00%          | 98.64%          | 1.36%                   | Buen ajuste (Generaliza) |
| KNN Regressor (k=5)         | 98.47%           | 97.95%          | 0.51%                   | Buen ajuste (Generaliza) |

=== 2. ANÁLISIS DE ERRORES EN TEST ===

Modelo: Regresión Lineal Múltiple | Total errores: 119 de 440 muestras
| Real       | Predicho    |   Cantidad de errores |
|:-----------|:------------|----------------------:|